<a href="https://colab.research.google.com/github/PYMaksim/ai-matrix-test-comparative-analysis/blob/main/ai_matrix_test_comparative_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:

"""
Project: LLM Comparative Analysis (Personalized Models)
Description: Testing 3 specific models available in your Yandex Cloud Folder.
Проект: Сравнение ИИ моделей (Персонализированные модели)
Описание: Тестирование 3 конкретных моделей, доступных в вашем каталоге Yandex Cloud.
"""

# ==============================================================================
# 1. IMPORTS (ИМПОРТ БИБЛИОТЕК)
# ==============================================================================
# We import 'userdata' to securely retrieve secrets from Google Colab environment.
# 'openai' is the official client library to interact with the API.
# 'time' is used to measure the response latency (speed) of each model.
# Импортируем 'userdata' для безопасного доступа к секретам в Colab.
# 'openai' — клиент для работы с API.
# 'time' — замеряем задержку (скорость) ответа каждой модели.
from google.colab import userdata
from openai import OpenAI
import time

# ==============================================================================
# 2. CONFIGURATION (НАСТРОЙКИ)
# ==============================================================================
# Retrieve API Key from Colab Secrets. This is the safest way to handle credentials.
# Получаем API-ключ из "Секретов" Colab. Это самый безопасный способ хранения токенов.
API_KEY = userdata.get('Y')

# Your specific Folder ID in Yandex Cloud. This is where the models are located.
# Твой идентификатор каталога в Yandex Cloud.
FOLDER_ID = "b1g2hgrbnnrqll94nr9k"

# The base URL for Yandex Cloud AI Studio API.
# Базовый адрес API-гейтвея Яндекса.
BASE_URL = "https://ai.api.cloud.yandex.net/v1"

# Headers for authentication. 'x-folder-id' is mandatory for Yandex API.
# Заголовки для авторизации. 'x-folder-id' обязателен для Яндекса.
HEADERS = {
    "Authorization": f"Api-Key {API_KEY}",
    "x-folder-id": FOLDER_ID
}

# Initialize the OpenAI client. We point it to Yandex endpoint.
# Инициализируем клиента OpenAI, указывая адрес сервера Яндекса.
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# ==============================================================================
# 3. UNIVERSAL PROMPT (ЕДИНЫЙ ПРОМПТ)
# ==============================================================================
# The question is in English to ensure a fair comparison across models.
# Вопрос на английском для честного сравнения (исключаем фактор родного языка).
USER_QUESTION = """Explain how a neural network learns as if you are talking to a 7-year-old child.
Do not just give the answer. First, provide a step-by-step reasoning (Chain of Thought),
and then give the final simple explanation."""

# System prompt defines the ROLE. It is identical for all models.
# Системный промпт задает РОЛЬ. Он одинаков для всех моделей.
SYSTEM_PROMPT = """You are a world-class AI educator and explainer.
Your task is to explain complex technical concepts to a 7-year-old.
You must follow this structure strictly:
1. Step-by-Step Reasoning (How it works inside)
2. Final Simple Answer (The analogy)"""

# ==============================================================================
# 4. TEST MATRIX (МАТРИЦА ТЕСТИРОВАНИЯ)
# ==============================================================================
# List of models to test. 'id' is used for the API URI, 'name' is for display.
# Список моделей. 'id' — для ссылки, 'name' — для красивого вывода в консоль.
# UPDATED: Using exact IDs from your 'models.list()' output.
# ОБНОВЛЕНО: Используем точные ID из вашего вывода 'models.list()'.
models_to_test = [
    {"id": "yandexgpt-5-pro", "name": "YandexGPT 5 Pro"},
    {"id": "deepseek-v4-flash", "name": "DeepSeek V4 Flash"},
    {"id": "qwen3.6-35b-a3b", "name": "Qwen 3.6 35B"}
]

# Temperature settings. 0.0 is deterministic (Logic), 0.8 is creative (Storytelling).
# Настройки температуры. 0.0 — детерминизм (Логика), 0.8 — креатив (Сказки).
temperatures = [
    {"value": 0.0, "label": "LOGIC (Precise)"},
    {"value": 0.8, "label": "CREATIVE (Storytelling)"}
]

# ==============================================================================
# 5. CORE FUNCTION (ОСНОВНАЯ ФУНКЦИЯ)
# ==============================================================================
def run_test(model_name, model_id, temp_value, temp_label):
    """
    Executes a single API call for a specific model and temperature.
    Выполняет один запрос к API для конкретной модели и температуры.
    """
    # Constructing the full model URI (Uniform Resource Identifier).
    # Собираем полный путь к модели (URI).
    model_uri = f"gpt://{FOLDER_ID}/{model_id}/latest"

    # Visual separator in the console for better readability of results.
    # Разделитель в консоли для читаемости результатов.
    print(f"\n{'='*20} {model_name} | {temp_label} {'='*20}")

    start = time.time() # Start the timer to measure latency.

    try:
        # The actual API request.
        # Сам запрос к API.
        response = client.chat.completions.create(
            model=model_uri,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_QUESTION}
            ],
            temperature=temp_value,
            # TECHNICAL FIX: Increased from 400 to 800 to prevent 'None' from heavy models.
            # ТЕХНИЧЕСКОЕ ИСПРАВЛЕНИЕ: Увеличено с 400 до 800, чтобы тяжелые модели не сгорали по токенам.
            max_tokens=800,
            extra_headers=HEADERS
        )

        latency = time.time() - start # Calculate how long the model thought.
        answer = response.choices[0].message.content

        # Output the results.
        # Выводим результат.
        print(f"[SUCCESS] Latency: {latency:.2f}s")
        print("-" * 10 + " ANSWER " + "-" * 10)
        print(answer)
        return {"model": model_name, "temp": temp_label, "answer": answer, "latency": latency}

    except Exception as e:
        # Error handling if the API call fails.
        # Обработка ошибок, если запрос упал.
        print(f"[SKIPPED/ERROR] {model_name} @ {temp_label} is not responding or access denied.")
        print(f"Details: {e}")
        return {"model": model_name, "temp": temp_label, "status": "SKIPPED"}

# ==============================================================================
# 6. EXECUTION (ВЫПОЛНЕНИЕ)
# ==============================================================================
print("🚀 Starting Personalized Matrix Test...")
results = []

# Nested loop: For every model, we run it through every temperature.
# Вложенный цикл: Для каждой модели мы прогоняем её через каждую температуру.
for model in models_to_test:
    for temp in temperatures:
        result = run_test(
            model["name"],
            model["id"],
            temp["value"],
            temp["label"]
        )
        results.append(result)

# ==============================================================================
# 7. SUMMARY REPORT (ИТОГОВЫЙ ОТЧЕТ)
# ==============================================================================
print("\n\n" + "="*60)
print("📊 SUMMARY REPORT")
print("="*60)
for r in results:
    if "status" in r and r["status"] == "SKIPPED":
        print(f"⚠️ SKIPPED: {r['model']} @ {r['temp']}")
    else:
        print(f"✅ OK: {r['model']} @ {r['temp']} (Latency: {r['latency']:.2f}s)")

🚀 Starting Personalized Matrix Test...

==================== YandexGPT 5 Pro | LOGIC (Precise) ====================
[SUCCESS] Latency: 8.30s
---------- ANSWER ----------
**Step-by-Step Reasoning:**

1. Imagine a neural network as a group of friendly robots that work together to learn new things.
2. These robots are given a task, like learning to recognize pictures of cats.
3. At first, the robots don't know what a cat looks like, so they make guesses based on the information they have.
4. When they guess wrong, they get a little signal that says, "Try again!" This helps them understand what they did wrong.
5. The robots then adjust their thinking a little bit to try to do better next time. It's like when you learn to ride a bike and fall down, you try again and get better.
6. They keep doing this over and over, getting better and better at recognizing cats.
7. After a lot of practice, the robots get really good at knowing what a cat looks like, and they can even tell different cats apa